# TNG100 <-> FIREbox domain shift -- 20-run variance check per method

Two fixes applied on top of the original notebook (see `scripts/process_data.py`'s
`sample_pyg_subgraph` and `ShiftKit/shiftkit/methods/sidda.py`):

1. **Train-test-split fix** -- `sample_pyg_subgraph` used to discard each sampled
   node's original train/val/test membership and re-assign a fresh, UNSTRATIFIED
   split from scratch. It now inherits the original (mass-stratified) split
   membership directly, the same way `induced_subgraph_by_mask` below already did.
2. **Softmax / potentials fix** -- `geomloss.SamplesLoss` returns Sinkhorn
   potentials shaped `(1, N)`. `.softmax(dim=0)` on that shape is a no-op (all
   weights come out equal to 1), so `SIDDA + OT-reweight`'s EMA branch was
   unreachable (its readiness check compared `shape[0]==1` to `shape[0]==N`,
   always False) and `SIDDA + Loss-reweight`'s per-sample weighting had zero
   effect on the gradient. Fixed to `.softmax(dim=-1)` plus a `.reshape(-1)`
   before feeding the weights back into `sinkhorn()` (which expects `(N,)`/`(N,1)`,
   not `(1,N)`).

Everything else is untouched from the original: 150 epochs, no gradient
clipping, no best-epoch selection (last-epoch is reported, same as the original
notebook), no seeding. **tng_to_fire only** (kept consistent with the earlier
3-run ablation steps), and `USE_FULL_TNG_GRAPH = True` by default -- the earlier
3-run comparison showed the full 7,228-node TNG graph cuts run-to-run variance
roughly in half versus the size-matched 1,035-node graph; flip the toggle below
to reproduce the size-matched version instead.

20 independent (unseeded) runs per method -- same rationale as the earlier 3- and
5-run steps: the original pipeline has no seeding, so each run is a fresh draw.
20 draws gives a per-node mean/std across runs (see the scatter plot at the end)
that isn't dominated by a couple of outlier draws the way 5 runs sometimes was.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm

REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / "scripts").exists() and (candidate / "data").exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate repository root containing 'scripts' and 'data'.")

SCRIPTS_DIR = REPO_ROOT / "scripts"
DM_ROOT = REPO_ROOT / "data"
SHIFTKIT_DIR = REPO_ROOT / "ShiftKit"

for path in [str(REPO_ROOT), str(SCRIPTS_DIR), str(DM_ROOT), str(SHIFTKIT_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from process_data import DataProcessor, sample_pyg_subgraph
from process_tng_data import TNGDataProcessor

OUTPUT_DIR = REPO_ROOT / "output" / "domain-shift-tng100-5runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "fire_path": DM_ROOT / "firebox_data" / "FIREbox_z=0.txt",
    "tng_path": DM_ROOT / "tng-data" / "TNG100" / "subhalos_99.parquet",
    "r": 1,
    "test_size": 0.1,
    "val_size": 0.1,
    "standardize": True,
    "stratify_bins": 10,
    "random_state": 42,
    "upper_mass": 12,
}
GRAPH_KWARGS = dict(
    r=CONFIG["r"], test_size=CONFIG["test_size"], val_size=CONFIG["val_size"],
    standardize=CONFIG["standardize"], stratify_bins=CONFIG["stratify_bins"],
    random_state=CONFIG["random_state"], upper_mass=CONFIG["upper_mass"],
)

print(f"Repo: {REPO_ROOT}")
print(f"TNG100 source file: {CONFIG['tng_path'].name}")
print(f"CUDA available: {torch.cuda.is_available()}"
      + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))


## Load both domains and build the overlap-restricted subgraph

In [ ]:
USE_FULL_TNG_GRAPH = True  # False reproduces the size-matched (1,035-node) TNG graph instead

fire_proc = DataProcessor(file_path=str(CONFIG["fire_path"]), subhalos="both")
fire_data = fire_proc.create_graph_data(**GRAPH_KWARGS)

tng_proc = TNGDataProcessor(file_path=str(CONFIG["tng_path"]), subhalos="both")

MAX_TNG_ROWS_FOR_GRAPH = 7500
if len(tng_proc.df_filtered) > MAX_TNG_ROWS_FOR_GRAPH:
    rng_pre = np.random.default_rng(CONFIG["random_state"])
    keep_idx = np.sort(rng_pre.choice(len(tng_proc.df_filtered), size=MAX_TNG_ROWS_FOR_GRAPH, replace=False))
    print(f"Pre-subsampling TNG100 catalog: {len(tng_proc.df_filtered):,} -> {MAX_TNG_ROWS_FOR_GRAPH:,} rows "
          f"(before graph construction, to keep radius-graph edge count tractable)")
    tng_proc.df_filtered = tng_proc.df_filtered.iloc[keep_idx].reset_index(drop=True)

tng_data = tng_proc.create_graph_data(**GRAPH_KWARGS)

FIRE_MSTAR_COL = "lg_Mstar_<Rhalo"
TNG_MSTAR_COL = "stellar_mass"
fdf = fire_proc.df_filtered
tdf_full = tng_proc.df_filtered
fire_mstar = fdf[FIRE_MSTAR_COL].values.astype(float)
tng_mstar = tdf_full[TNG_MSTAR_COL].values.astype(float)
overlap_lo, overlap_hi = tng_mstar.min(), fire_mstar.max()
print(f"Overlap range: [{overlap_lo:.2f}, {overlap_hi:.2f}] dex in lg_Mstar")

import torch
from torch_geometric.data import Data


def induced_subgraph_by_mask(data, keep_mask):
    '''Induced subgraph over nodes where keep_mask is True.'''
    keep_idx = keep_mask.nonzero(as_tuple=False).view(-1)
    index_map = {int(old): new for new, old in enumerate(keep_idx.tolist())}
    keep_set = set(keep_idx.tolist())

    edge_index = data.edge_index
    src, dst = edge_index[0].tolist(), edge_index[1].tolist()
    edge_mask = torch.tensor([s in keep_set and d in keep_set for s, d in zip(src, dst)], dtype=torch.bool)
    kept_edges = edge_index[:, edge_mask]
    remapped = torch.stack([
        torch.tensor([index_map[int(s)] for s in kept_edges[0].tolist()], dtype=torch.long),
        torch.tensor([index_map[int(d)] for d in kept_edges[1].tolist()], dtype=torch.long),
    ])

    sub = Data(
        x=data.x[keep_idx],
        edge_index=remapped,
        y=data.y[keep_idx],
        pos=data.pos[keep_idx] if getattr(data, "pos", None) is not None else None,
    )
    for split in ("train_mask", "val_mask", "test_mask"):
        if hasattr(data, split):
            setattr(sub, split, getattr(data, split)[keep_idx])
    return sub


fire_keep = torch.tensor((fdf[FIRE_MSTAR_COL].values >= overlap_lo) & (fdf[FIRE_MSTAR_COL].values <= overlap_hi))
tng_keep = torch.tensor((tdf_full[TNG_MSTAR_COL].values >= overlap_lo) & (tdf_full[TNG_MSTAR_COL].values <= overlap_hi))

fire_overlap_graph = induced_subgraph_by_mask(fire_data, fire_keep)
tng_overlap_full_graph = induced_subgraph_by_mask(tng_data, tng_keep)
tng_overlap_matched_graph = sample_pyg_subgraph(
    tng_overlap_full_graph, num_nodes_sample=fire_overlap_graph.num_nodes, random_state=CONFIG["random_state"]
)
tng_graph = tng_overlap_full_graph if USE_FULL_TNG_GRAPH else tng_overlap_matched_graph
tng_graph_label = "TNG100 (overlap, full)" if USE_FULL_TNG_GRAPH else "TNG100 (overlap, size-matched)"

for name, d in [
    ("FIREbox (overlap)", fire_overlap_graph),
    (tng_graph_label, tng_graph),
]:
    print(f"{name:32s} nodes={d.num_nodes:,}  edges={d.num_edges:,}  "
          f"train={int(d.train_mask.sum())}  val={int(d.val_mask.sum())}  test={int(d.test_mask.sum())}")

# tng_to_fire only, consistent with the earlier 3-run ablation steps.
DIRECTIONS = {
    "tng_to_fire": dict(src=tng_graph, tgt=fire_overlap_graph, src_name="TNG100", tgt_name="FIREbox"),
}


## Train each of the 4 methods 20 times (tng_to_fire only)

In [ ]:
import json
import time

import numpy as np

from shiftkit import (
    GNN, DataManager,
    SourceOnlyGaussianRegressionTrainer, SIDDAGaussianRegressionTrainer,
)

MODEL_CONFIG = {"hidden_channels": 64, "num_layers": 2, "model_name": "SAGE"}
EPOCHS = 150
WARMUP = 10
LR = 5e-3
N_RUNS = 20

METHOD_SPECS = [
    dict(label="No-DA",                    kind="sourceonly"),
    dict(label="SIDDA",                    kind="sidda", use_potentials=False, weight_ot=False),
    dict(label="SIDDA + OT-reweight",      kind="sidda", use_potentials=False, weight_ot=True),
    dict(label="SIDDA + Loss-reweight",    kind="sidda", use_potentials=True,  weight_ot=False),
]
ALL_METHOD_ORDER = [m["label"] for m in METHOD_SPECS]


def build_trainer(spec, model, train_src, train_tgt):
    if spec["kind"] == "sourceonly":
        return SourceOnlyGaussianRegressionTrainer(model, train_src, train_tgt, lr=LR)
    if spec["kind"] == "sidda":
        return SIDDAGaussianRegressionTrainer(
            model, train_src, train_tgt, lr=LR, warmup_epochs=WARMUP,
            use_potentials=spec["use_potentials"], weight_ot=spec["weight_ot"],
            potential_temperature=1.0, ot_ema_momentum=0.9,
        )
    raise ValueError(spec["kind"])


def compute_chi2(true, mean, std):
    pulls = (mean - true) / std
    return float(np.mean(pulls ** 2))


all_results = {label: [] for label in ALL_METHOD_ORDER}  # label -> list of per-run result dicts

for direction, spec in DIRECTIONS.items():
    src, tgt = spec["src"], spec["tgt"]
    ckpt_dir = OUTPUT_DIR / f"checkpoints_overlap_{direction}_4methods"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    progress_log = OUTPUT_DIR / f"overlap_{direction}_4methods_progress.log"

    def log_progress(msg):
        line = f"[{time.strftime('%H:%M:%S')}] {msg}"
        print(line)
        with open(progress_log, "a") as f:
            f.write(line + "\n")

    dm = DataManager(batch_size=1, num_workers=0)
    train_src, train_tgt = dm.load("pyg_domains", train=True, source=src, target=tgt)
    test_src, test_tgt = dm.load("pyg_domains", train=False, source=src, target=tgt)

    for method_spec in METHOD_SPECS:
        label = method_spec["label"]
        for run in range(N_RUNS):
            log_progress(f"{direction} | {label} | run {run} : starting")
            t0 = time.time()

            model = GNN(
                src, MODEL_CONFIG["model_name"],
                hidden_channels=MODEL_CONFIG["hidden_channels"],
                num_layers=MODEL_CONFIG["num_layers"],
                regress=True, pool="none", predict_var=True,
            )
            trainer = build_trainer(method_spec, model, train_src, train_tgt)
            trainer.fit(epochs=EPOCHS)
            elapsed = time.time() - t0

            r_src = trainer.evaluate(test_src, domain="source")
            r_tgt = trainer.evaluate(test_tgt, domain="target")
            true_src, mean_src, std_src = trainer.predict(test_src)
            true_tgt, mean_tgt, std_tgt = trainer.predict(test_tgt)
            chi2_src = compute_chi2(true_src, mean_src, std_src)
            chi2_tgt = compute_chi2(true_tgt, mean_tgt, std_tgt)

            run_result = {
                "run": run, "direction": direction,
                "src_rmse": r_src["rmse"], "src_r2": r_src["r2"], "src_chi2": chi2_src,
                "tgt_rmse": r_tgt["rmse"], "tgt_r2": r_tgt["r2"], "tgt_chi2": chi2_tgt,
                "elapsed_s": elapsed,
            }
            all_results[label].append(run_result)

            np.savez(
                ckpt_dir / f"{label.replace(' ', '_')}_run{run}_predictions.npz",
                true_src=true_src, mean_src=mean_src, std_src=std_src,
                true_tgt=true_tgt, mean_tgt=mean_tgt, std_tgt=std_tgt,
            )
            log_progress(
                f"{direction} | {label} | run {run} : done in {elapsed:6.1f}s  "
                f"src RMSE={r_src['rmse']:.4f} R2={r_src['r2']:.3f} chi2={chi2_src:.3f}  |  "
                f"tgt RMSE={r_tgt['rmse']:.4f} R2={r_tgt['r2']:.3f} chi2={chi2_tgt:.3f}"
            )

    log_progress(f"=== direction {direction} complete ===")

with open(OUTPUT_DIR / "all_results_5runs.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nsaved", OUTPUT_DIR / "all_results_5runs.json")


## Per-run results and mean +/- std summary

In [ ]:
import pandas as pd

METRICS = ["src_rmse", "src_r2", "src_chi2", "tgt_rmse", "tgt_r2", "tgt_chi2"]

print("Per-run results\n" + "=" * 100)
rows = []
for label in ALL_METHOD_ORDER:
    for r in all_results[label]:
        rows.append({"method": label, **{k: r[k] for k in ["run"] + METRICS}})
per_run_df = pd.DataFrame(rows)
csv_path = OUTPUT_DIR / "per_run_results.csv"
per_run_df.to_csv(csv_path, index=False)

for label in ALL_METHOD_ORDER:
    print(f"\n--- {label} ---")
    sub = per_run_df[per_run_df["method"] == label].set_index("run")
    print(sub[METRICS].to_string(float_format=lambda v: f"{v:.4f}"))

print("\n" + "=" * 100)
print(f"Mean +/- std across {len(all_results[ALL_METHOD_ORDER[0]])} runs\n" + "=" * 100)
summary_rows = []
for label in ALL_METHOD_ORDER:
    sub = per_run_df[per_run_df["method"] == label]
    row = {"method": label}
    for m in METRICS:
        row[f"{m}_mean"] = sub[m].mean()
        row[f"{m}_std"] = sub[m].std()
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows).set_index("method")
summary_df.to_csv(OUTPUT_DIR / "summary_mean_std.csv")

for label in ALL_METHOD_ORDER:
    row = summary_df.loc[label]
    print(f"\n{label}:")
    print(f"  src: RMSE={row['src_rmse_mean']:.4f}+-{row['src_rmse_std']:.4f}  "
          f"R2={row['src_r2_mean']:.3f}+-{row['src_r2_std']:.3f}  "
          f"chi2={row['src_chi2_mean']:.3f}+-{row['src_chi2_std']:.3f}")
    print(f"  tgt: RMSE={row['tgt_rmse_mean']:.4f}+-{row['tgt_rmse_std']:.4f}  "
          f"R2={row['tgt_r2_mean']:.3f}+-{row['tgt_r2_std']:.3f}  "
          f"chi2={row['tgt_chi2_mean']:.3f}+-{row['tgt_chi2_std']:.3f}")

print("\nsaved", csv_path)
print("saved", OUTPUT_DIR / "summary_mean_std.csv")


## Comparison plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
colors = {"No-DA": "#4C72B0", "SIDDA": "#DD8452", "SIDDA + OT-reweight": "#55A868",
          "SIDDA + Loss-reweight": "#C44E52"}

for ax, (metric, ylabel) in zip(axes, [("tgt_rmse", "target RMSE"), ("tgt_r2", "target R$^2$")]):
    for x, label in enumerate(ALL_METHOD_ORDER):
        vals = [r[metric] for r in all_results[label]]
        mean, std = np.mean(vals), np.std(vals)
        jitter = np.random.default_rng(0).normal(0, 0.05, size=len(vals))
        ax.scatter(np.full(len(vals), x) + jitter, vals, s=20, alpha=0.4, color=colors[label], zorder=2)
        ax.errorbar(x, mean, yerr=std, fmt="o", ms=10, capsize=6, lw=2.2,
                    color=colors[label], mec="k", mew=0.8, zorder=3)
    ax.set_xticks(range(len(ALL_METHOD_ORDER)))
    ax.set_xticklabels(ALL_METHOD_ORDER, rotation=20, ha="right", fontsize=8)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25, lw=0.5)
    if metric == "tgt_r2":
        ax.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)

fig.suptitle(f"tng_to_fire, {N_RUNS} runs per method -- split-fix + softmax-fix applied", fontsize=11)
fig.tight_layout()
fig_path = OUTPUT_DIR / "tng_to_fire_5runs_comparison.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print("saved", fig_path)
plt.show()


## Predicted vs. true, aggregated across all 20 runs

For each test node, the SAME physical halo is predicted 20 times (the data
split is fixed -- only training stochasticity varies). This plots, per node,
the MEAN prediction across the 20 runs against the true value, with error bars
= the STD across runs -- i.e. how much a given halo's predicted mass actually
varies from one training run to the next, not the model's own per-run
uncertainty head. Same layout as the original notebook's pred-vs-true plot
(top row = target/FIREbox, bottom row = source/TNG100), one panel per method,
colored the same as the summary plots.


In [ ]:
METHOD_COLORS = {"No-DA": "#4C72B0", "SIDDA": "#DD8452", "SIDDA + OT-reweight": "#55A868",
                  "SIDDA + Loss-reweight": "#C44E52"}

direction = "tng_to_fire"
ckpt_dir = OUTPUT_DIR / f"checkpoints_overlap_{direction}_4methods"

fig, axes = plt.subplots(2, len(ALL_METHOD_ORDER), figsize=(16, 8), sharex=True, sharey=True)

for col, label in enumerate(ALL_METHOD_ORDER):
    color = METHOD_COLORS[label]
    fname = label.replace(" ", "_")

    true_tgt_runs, mean_tgt_runs = [], []
    true_src_runs, mean_src_runs = [], []
    for run in range(N_RUNS):
        z = np.load(ckpt_dir / f"{fname}_run{run}_predictions.npz")
        true_tgt_runs.append(z["true_tgt"]); mean_tgt_runs.append(z["mean_tgt"])
        true_src_runs.append(z["true_src"]); mean_src_runs.append(z["mean_src"])

    true_tgt_runs = np.stack(true_tgt_runs)   # (N_RUNS, n_test_tgt)
    mean_tgt_runs = np.stack(mean_tgt_runs)
    true_src_runs = np.stack(true_src_runs)
    mean_src_runs = np.stack(mean_src_runs)

    # The test split is fixed across runs (no data-split seeding), so the true
    # values should be identical run to run -- verify rather than assume.
    assert np.allclose(true_tgt_runs, true_tgt_runs[0]), f"{label}: target test labels differ across runs"
    assert np.allclose(true_src_runs, true_src_runs[0]), f"{label}: source test labels differ across runs"
    true_tgt, true_src = true_tgt_runs[0], true_src_runs[0]

    tgt_run_mean, tgt_run_std = mean_tgt_runs.mean(axis=0), mean_tgt_runs.std(axis=0)
    src_run_mean, src_run_std = mean_src_runs.mean(axis=0), mean_src_runs.std(axis=0)

    for row, (true, run_mean, run_std, dom_name) in enumerate([
        (true_tgt, tgt_run_mean, tgt_run_std, "FIREbox (target)"),
        (true_src, src_run_mean, src_run_std, "TNG100 (source)"),
    ]):
        ax = axes[row, col]
        lo, hi = true.min(), true.max()
        ax.plot([lo, hi], [lo, hi], color="k", lw=1, ls="--", alpha=0.5, zorder=1)
        ax.errorbar(true, run_mean, yerr=run_std, fmt="o", ms=3, elinewidth=0.6,
                    alpha=0.5, color=color, ecolor=color, zorder=2)
        ss_res = ((run_mean - true) ** 2).sum()
        ss_tot = ((true - true.mean()) ** 2).sum()
        r2_of_mean = 1 - ss_res / ss_tot
        mean_run_std = run_std.mean()
        ax.set_title(f"{label}\n{dom_name}: R$^2$={r2_of_mean:.3f}  mean std={mean_run_std:.3f}", fontsize=8.5)
        if col == 0:
            ax.set_ylabel(("Target" if row == 0 else "Source") + "\nmean predicted $\\log M_{halo}/M_\\odot$")
        if row == 1:
            ax.set_xlabel("true $\\log M_{halo}/M_\\odot$")

plt.suptitle(f"Predicted vs. true halo mass -- mean +/- std across {N_RUNS} runs, {direction}",
             fontweight="bold")
plt.tight_layout()
fig_path = OUTPUT_DIR / f"{direction}_{N_RUNS}runs_mean_std_scatter.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
print("saved", fig_path)
plt.show()
